<a href="https://colab.research.google.com/github/alexsiks/estudo_estatistica/blob/main/An%C3%A1lise_de_Distribui%C3%A7%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import numpy as np
import pandas as pd
import plotly.express as px

Modelo de Distribuição Normal

In [14]:


# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

inicio = "2026-01-01 08:00:00"

tempo_total_minutos = 30
intervalo_segundos = 2
numero_bins = 10

# Quantidade de registros
periodos = (
    tempo_total_minutos * 60
) // intervalo_segundos


# ============================================================
# 2. CRIAR DATAFRAME
# ============================================================

np.random.seed(42)

df = pd.DataFrame({
    "datetime": pd.date_range(
        start=inicio,
        periods=periodos,
        freq="2s"
    ),

    "falha": np.random.choice(
        [0, 1],
        size=periodos,
        p=[0.95, 0.05]
    )
})


# ============================================================
# 3. TEMPO DECORRIDO EM MINUTOS
# ============================================================

df["tempo_minutos"] = (
    df["datetime"] - df["datetime"].min()
).dt.total_seconds() // 60


# ============================================================
# 4. CRIAR CLASSIFICAÇÃO
# ============================================================

# Cada faixa terá 3 minutos
tamanho_bin = tempo_total_minutos // numero_bins

df["bin"] = (
    df["tempo_minutos"] // tamanho_bin
) + 1


# Evita criar um bin 11 caso exista exatamente 30 minutos
df["bin"] = df["bin"].clip(
    upper=numero_bins
)


# ============================================================
# 5. CRIAR RÓTULOS
# ============================================================

nomes_bins = []

for i in range(numero_bins):

    inicio_faixa = i * tamanho_bin

    # Próxima faixa começa no número seguinte
    fim_faixa = inicio_faixa + tamanho_bin - 1

    nomes_bins.append(
        f"{inicio_faixa}-{fim_faixa}"
    )


# Associar nome da faixa
df["faixa"] = df["bin"].map(
    dict(
        enumerate(
            nomes_bins,
            start=1
        )
    )
)


# ============================================================
# 6. CONTAR FALHAS POR FAIXA
# ============================================================

df = (
    df
    .groupby(
        ["bin", "faixa"],
        as_index=False
    )
    .agg(
        quantidade_falhas=("falha", "sum"),
        quantidade_registros=("falha", "size")
    )
)


# ============================================================
# 7. GARANTIR TODOS OS 10 BINS
# ============================================================

df = (
    pd.DataFrame({
        "bin": range(1, numero_bins + 1),
        "faixa": nomes_bins
    })
    .merge(
        df,
        on=["bin", "faixa"],
        how="left"
    )
    .fillna(0)
)


# ============================================================
# 8. RESULTADO
# ============================================================

print(df)

   bin  faixa  quantidade_falhas  quantidade_registros
0    1    0-2                  5                    90
1    2    3-5                  3                    90
2    3    6-8                  5                    90
3    4   9-11                  3                    90
4    5  12-14                  6                    90
5    6  15-17                 12                    90
6    7  18-20                  2                    90
7    8  21-23                  5                    90
8    9  24-26                  0                    90
9   10  27-29                  3                    90


In [15]:
import numpy as np
import plotly.graph_objects as go
from scipy import stats


# ============================================================
# 2. BINS
# ============================================================

# Assuming the intent is to analyze the 'quantidade_falhas' column.
df = df['quantidade_falhas']

bin = 10

minimo = df.min()
maximo = df.max()

limites = np.linspace(
    minimo,
    maximo + 1,
    bin + 1
)

limites = np.floor(limites).astype(int)

# ============================================================
# 3. CLASSIFICAÇÃO
# ============================================================

classes = np.digitize(
    df,
    limites[1:-1]
)

# ============================================================
# 4. NOMES DOS INTERVALOS
# ============================================================

nomes = [
    f"{limites[i]}-{limites[i + 1] - 1}"
    for i in range(len(limites) - 1)
]

# ============================================================
# 5. FREQUÊNCIAS
# ============================================================

frequencias = [
    np.sum(classes == i)
    for i in range(bin)
]

# ============================================================
# 6. CENTROS DOS BINS
# ============================================================

centros = (
    limites[:-1] + limites[1:] - 1
) / 2

# ============================================================
# 7. ESTATÍSTICAS
# ============================================================

media = np.mean(df)

mediana = np.median(df)

valores, contagens = np.unique(
    df,
    return_counts=True
)

moda = valores[np.argmax(contagens)]

variancia = np.var(df)

desvio_padrao = np.std(df)

# Excesso de curtose
curtose = stats.kurtosis(
    df,
    fisher=True
)

# Assimetria
assimetria = stats.skew(df)

# ============================================================
# 8. CLASSIFICAÇÃO DA CURTOSE
# ============================================================

if curtose > 0:
    tipo_curtose = "Leptocúrtica"
elif curtose < 0:
    tipo_curtose = "Platicúrtica"
else:
    tipo_curtose = "Mesocúrtica"

# ============================================================
# 9. CLASSIFICAÇÃO DA ASSIMETRIA
# ============================================================

if assimetria > 0:
    tipo_assimetria = "Assimetria positiva"
elif assimetria < 0:
    tipo_assimetria = "Assimetria negativa"
else:
    tipo_assimetria = "Simétrica"

# ============================================================
# 10. BARRAS
# ============================================================

bar_trace = go.Bar(
    x=centros,
    y=frequencias,
    text=frequencias,
    textposition="outside",
    name="Frequência",
    customdata=nomes,
    hovertemplate=(
        "Intervalo: %{customdata}<br>"
        "Frequência: %{y}"
        "<extra></extra>"
    )
)

fig = go.Figure()

fig.add_trace(bar_trace)

# ============================================================
# 11. CURVA DE DISTRIBUIÇÃO
# ============================================================

x_curva = np.linspace(
    minimo,
    maximo,
    500
)

kde = stats.gaussian_kde(df)

densidade = kde(x_curva)

largura_bin = np.mean(
    np.diff(limites)
)

curva = (
    densidade
    * len(df)
    * largura_bin
)

fig.add_trace(
    go.Scatter(
        x=x_curva,
        y=curva,
        mode="lines",
        name="Curva de distribuição",
        line=dict(
            width=3
        )
    )
)

# ============================================================
# 12. LINHA DA MÉDIA
# ============================================================

fig.add_vline(
    x=media,
    line=dict(
        color="blue",
        dash="dash",
        width=3
    ),
    annotation_text=f"Média = {media:.2f}",
    annotation_position="top"
)

# ============================================================
# 13. LINHA DA MEDIANA
# ============================================================

fig.add_vline(
    x=mediana,
    line=dict(
        color="green",
        dash="dot",
        width=3
    ),
    annotation_text=f"Mediana = {mediana:.2f}",
    annotation_position="bottom"
)

# ============================================================
# 14. LINHA DA MODA
# ============================================================

fig.add_vline(
    x=moda,
    line=dict(
        color="red",
        dash="dashdot",
        width=3
    ),
    annotation_text=f"Moda = {moda:.2f}",
    annotation_position="top left"
)

# ============================================================
# 15. EIXO X
# ============================================================

fig.update_xaxes(
    tickmode="array",
    tickvals=centros,
    ticktext=nomes
)

# ============================================================
# 16. RÉGUA DA ASSIMETRIA
# ============================================================

y_assimetria = max(frequencias) + 3

assim_min = -2
assim_max = 2

# Linha principal
fig.add_shape(
    type="line",
    x0=assim_min,
    x1=assim_max,
    y0=y_assimetria,
    y1=y_assimetria,
    line=dict(
        color="black",
        width=2
    )
)

# Marcações da régua
for valor in np.arange(
    assim_min,
    assim_max + 0.1,
    0.5
):

    fig.add_shape(
        type="line",
        x0=valor,
        x1=valor,
        y0=y_assimetria - 0.3,
        y1=y_assimetria + 0.3,
        line=dict(
            color="black",
            width=1
        )
    )

    fig.add_annotation(
        x=valor,
        y=y_assimetria + 0.7,
        text=f"{valor:g}",
        showarrow=False,
        font=dict(
            size=10
        )
    )

# Marcador da assimetria
fig.add_trace(
    go.Scatter(
        x=[assimetria],
        y=[y_assimetria],
        mode="markers+text",
        marker=dict(
            size=14,
            color="purple",
            symbol="diamond"
        ),
        text=[
            f"{assimetria:.2f}"
        ],
        textposition="top center",
        name="Assimetria",
        hovertemplate=(
            f"Assimetria: {assimetria:.2f}<br>"
            f"Classificação: {tipo_assimetria}"
            "<extra></extra>"
        )
    )
)

# Título da régua
fig.add_annotation(
    x=assim_min,
    y=y_assimetria + 1.2,
    text="<b>Régua da Assimetria</b>",
    showarrow=False,
    xanchor="left"
)

# Descrição
fig.add_annotation(
    x=0,
    y=y_assimetria - 0.8,
    text=(
        "← Negativa        0 = Simétrica        Positiva →"
    ),
    showarrow=False,
    font=dict(
        size=10
    )
)

# ============================================================
# 17. RÉGUA DA CURTOSE
# ============================================================

y_curtose = y_assimetria + 3

curt_min = -3
curt_max = 3

# Linha principal
fig.add_shape(
    type="line",
    x0=curt_min,
    x1=curt_max,
    y0=y_curtose,
    y1=y_curtose,
    line=dict(
        color="black",
        width=2
    )
)

# Marcações da régua
for valor in np.arange(
    curt_min,
    curt_max + 0.1,
    0.5
):

    fig.add_shape(
        type="line",
        x0=valor,
        x1=valor,
        y0=y_curtose - 0.3,
        y1=y_curtose + 0.3,
        line=dict(
            color="black",
            width=1
        )
    )

    fig.add_annotation(
        x=valor,
        y=y_curtose + 0.7,
        text=f"{valor:g}",
        showarrow=False,
        font=dict(
            size=10
        )
    )

# ============================================================
# 18. MARCADOR DA CURTOSE
# ============================================================

fig.add_trace(
    go.Scatter(
        x=[curtose],
        y=[y_curtose],
        mode="markers+text",
        marker=dict(
            size=14,
            color="orange",
            symbol="diamond"
        ),
        text=[
            f"{curtose:.2f}<br>{tipo_curtose}"
        ],
        textposition="top center",
        name="Curtose",
        hovertemplate=(
            f"Curtose: {curtose:.2f}<br>"
            f"Tipo: {tipo_curtose}"
            "<extra></extra>"
        )
    )
)

# ============================================================
# 19. TÍTULO DA RÉGUA DE CURTOSE
# ============================================================

fig.add_annotation(
    x=curt_min,
    y=y_curtose + 1.2,
    text="<b>Régua da Curtose</b>",
    showarrow=False,
    xanchor="left"
)

# Descrição da curtose
fig.add_annotation(
    x=0,
    y=y_curtose - 0.8,
    text=(
        "← Platicúrtica     0 = Mesocúrtica     Leptocúrtica →"
    ),
    showarrow=False,
    font=dict(
        size=10
    )
)

# ============================================================
# 20. QUADRO DE ESTATÍSTICAS
# ============================================================

fig.add_annotation(
    x=1,
    y=0.98,
    xref="paper",
    yref="paper",
    xanchor="right",
    yanchor="top",
    align="left",

    text=(
        f"<b>Estatísticas</b><br>"
        f"Média = {media:.2f}<br>"
        f"Mediana = {mediana:.2f}<br>"
        f"Moda = {moda:.2f}<br>"
        f"Variância = {variancia:.2f}<br>"
        f"Desvio padrão = {desvio_padrao:.2f}<br>"
        f"Curtose = {curtose:.2f}<br>"
        f"<b>Tipo:</b> {tipo_curtose}<br>"
        f"Assimetria = {assimetria:.2f}<br>"
        f"<b>Tipo:</b> {tipo_assimetria}"
    ),

    showarrow=False,

    bgcolor="white",
    bordercolor="gray",
    borderwidth=1
)

# ============================================================
# 21. LAYOUT
# ============================================================

fig.update_layout(

    title=(
        "Distribuição dos dados — "
        "Curtose e Assimetria"
    ),

    xaxis_title="Intervalo",

    yaxis_title="Frequência",

    legend=dict(
        title="Legenda",
        orientation="v",
        x=1.02,
        y=0.5
    ),

    bargap=0.05,

    margin=dict(
        t=100,
        b=120,
        r=220,
        l=70
    ),

    height=800
)

# ============================================================
# 22. EXIBIR
# ============================================================

fig.show()

In [16]:

import numpy as np
import plotly.graph_objects as go
from scipy import stats

# ============================================================
# 1. DADOS
# ============================================================

df = np.array([
    1, 4, 8, 8, 8, 3, 6, 3, 4, 7, 9,
    11, 12, 15, 18, 18, 20, 21,
    25, 25, 28, 30,
    35, 38, 40,
    45, 47, 50,
    55, 58, 60
])

# ============================================================
# 2. BINS
# ============================================================

bin = 10

minimo = df.min()
maximo = df.max()

limites = np.linspace(
    minimo,
    maximo + 1,
    bin + 1
)

limites = np.floor(limites).astype(int)

# ============================================================
# 3. CLASSIFICAÇÃO
# ============================================================

classes = np.digitize(
    df,
    limites[1:-1]
)

# ============================================================
# 4. NOMES DOS INTERVALOS
# ============================================================

nomes = [
    f"{limites[i]}-{limites[i + 1] - 1}"
    for i in range(len(limites) - 1)
]

# ============================================================
# 5. FREQUÊNCIAS
# ============================================================

frequencias = [
    np.sum(classes == i)
    for i in range(bin)
]

# ============================================================
# 6. CENTROS DOS BINS
# ============================================================

centros = (
    limites[:-1] + limites[1:] - 1
) / 2

# ============================================================
# 7. ESTATÍSTICAS
# ============================================================

media = np.mean(df)

mediana = np.median(df)

valores, contagens = np.unique(
    df,
    return_counts=True
)

moda = valores[np.argmax(contagens)]

variancia = np.var(df)

desvio_padrao = np.std(df)

curtose = stats.kurtosis(
    df,
    fisher=True
)

assimetria = stats.skew(df)

# ============================================================
# 8. BARRAS
# ============================================================

bar_trace = go.Bar(
    x=centros,
    y=frequencias,
    text=frequencias,
    textposition="outside",
    name="Frequência",
    customdata=nomes,
    hovertemplate=(
        "Intervalo: %{customdata}<br>"
        "Frequência: %{y}"
        "<extra></extra>"
    )
)

fig = go.Figure()

fig.add_trace(bar_trace)

# ============================================================
# 9. CURVA DE DISTRIBUIÇÃO
# ============================================================

x_curva = np.linspace(
    minimo,
    maximo,
    500
)

# KDE baseado nos próprios dados
kde = stats.gaussian_kde(df)

densidade = kde(x_curva)

# Escala da curva para acompanhar a altura do histograma
largura_bin = np.mean(
    np.diff(limites)
)

curva = (
    densidade
    * len(df)
    * largura_bin
)

fig.add_trace(
    go.Scatter(
        x=x_curva,
        y=curva,
        mode="lines",
        name="Curva de distribuição",
        line=dict(
            width=3
        )
    )
)

# ============================================================
# 10. LINHAS DE MÉDIA, MEDIANA E MODA
# ============================================================

fig.add_vline(
    x=media,
    line=dict(
        color="blue",
        dash="dash",
        width=3
    ),
    annotation_text=f"Média = {media:.2f}",
    annotation_position="top"
)

fig.add_vline(
    x=mediana,
    line=dict(
        color="green",
        dash="dot",
        width=3
    ),
    annotation_text=f"Mediana = {mediana:.2f}",
    annotation_position="bottom"
)

fig.add_vline(
    x=moda,
    line=dict(
        color="red",
        dash="dashdot",
        width=3
    ),
    annotation_text=f"Moda = {moda:.2f}",
    annotation_position="top left"
)

# ============================================================
# 11. EIXO X
# ============================================================

fig.update_xaxes(
    tickmode="array",
    tickvals=centros,
    ticktext=nomes
)

# ============================================================
# 12. ESTATÍSTICAS
# ============================================================

fig.add_annotation(
    x=1,
    y=1,
    xref="paper",
    yref="paper",
    xanchor="right",
    yanchor="top",
    align="left",
    text=(
        f"<b>Estatísticas</b><br>"
        f"Média = {media:.2f}<br>"
        f"Mediana = {mediana:.2f}<br>"
        f"Moda = {moda:.2f}<br>"
        f"Variância = {variancia:.2f}<br>"
        f"Desvio padrão = {desvio_padrao:.2f}<br>"
        f"Curtose = {curtose:.2f}<br>"
        f"Assimetria = {assimetria:.2f}"
    ),
    showarrow=False,
    bgcolor="white",
    bordercolor="gray",
    borderwidth=1
)

# ============================================================
# 13. LAYOUT
# ============================================================

fig.update_layout(
    title="Distribuição dos dados",
    xaxis_title="Intervalo",
    yaxis_title="Frequência",
    legend=dict(
        title="Legenda",
        orientation="v",
        x=1.02,
        y=0.5
    ),
    bargap=0.05
)

fig.show()

Modelo de Distribuição de Falhas por Tempo

In [17]:

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.stats import gaussian_kde


# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

tempo_total_minutos = 20
numero_bins = 10


# ============================================================
# 2. GERAR O DATAFRAME
# ============================================================

df = pd.DataFrame({
    "datetime": pd.date_range(
        start="2026-01-01 08:00:00",
        periods=20,
        freq="1min"
    ),

    "falha": [
        0, 0, 1, 0, 1,
        0, 0, 0, 1, 0,
        1, 1, 1, 0, 1,
        0, 1, 1, 0, 1
    ]
})


# ============================================================
# 3. TEMPO DECORRIDO
# ============================================================

df["tempo_segundos"] = (
    df["datetime"] - df["datetime"].min()
).dt.total_seconds()

df["tempo_minutos"] = (
    df["tempo_segundos"] / 60
)


# ============================================================
# 4. CRIAR LIMITES NUMÉRICOS
# ============================================================

limites = np.linspace(
    0,
    tempo_total_minutos,
    numero_bins + 1
)


# ============================================================
# 5. CLASSIFICAR OS DADOS
# ============================================================

df["bin"] = pd.cut(
    df["tempo_minutos"],
    bins=limites,
    labels=False,
    include_lowest=True
)

df["bin"] = (
    df["bin"]
    .fillna(numero_bins - 1)
    .astype(int)
)


# ============================================================
# 6. CRIAR NOMES DAS CLASSES
#
# Exemplo:
# 0-2
# 3-5
# 6-8
# 9-11
# ...
# ============================================================

nomes_bins = []

inicio = 0

# Tamanho aproximado da classe
tamanho = int(
    np.ceil(
        (tempo_total_minutos + 1)
        / numero_bins
    )
)

for i in range(numero_bins):

    if i == numero_bins - 1:

        fim = tempo_total_minutos

    else:

        fim = inicio + tamanho - 1

        # Não ultrapassar o limite máximo
        fim = min(
            fim,
            tempo_total_minutos
        )

    nomes_bins.append(
        f"{inicio}-{fim}"
    )

    inicio = fim + 1

    if inicio > tempo_total_minutos:
        break


# ============================================================
# 7. GARANTIR EXATAMENTE O NÚMERO DE BINS
# ============================================================

while len(nomes_bins) < numero_bins:

    inicio = int(
        nomes_bins[-1].split("-")[1]
    ) + 1

    nomes_bins.append(
        f"{inicio}-{tempo_total_minutos}"
    )


# ============================================================
# 8. ASSOCIAR CLASSE AO DATAFRAME
# ============================================================

df["faixa"] = df["bin"].map(
    dict(
        enumerate(nomes_bins)
    )
)


# ============================================================
# 9. DISTRIBUIÇÃO DE FALHAS
# ============================================================

df_distribuicao = (
    df
    .groupby(
        ["bin", "faixa"],
        observed=False
    )["falha"]
    .sum()
    .reset_index(
        name="quantidade_falhas"
    )
)


# ============================================================
# 10. GARANTIR TODOS OS BINS
# ============================================================

bins_completos = pd.DataFrame({
    "bin": range(numero_bins),
    "faixa": nomes_bins
})

df_distribuicao = (
    bins_completos
    .merge(
        df_distribuicao,
        on=["bin", "faixa"],
        how="left"
    )
)

df_distribuicao[
    "quantidade_falhas"
] = (
    df_distribuicao[
        "quantidade_falhas"
    ]
    .fillna(0)
)


# ============================================================
# 11. CENTRO DAS CLASSES
# ============================================================

centros_bins = []

for faixa in nomes_bins:

    inicio, fim = map(
        int,
        faixa.split("-")
    )

    centro = (
        inicio + fim
    ) / 2

    centros_bins.append(
        centro
    )


# ============================================================
# 12. ESTATÍSTICAS
# ============================================================

tempos_falha = df.loc[
    df["falha"] == 1,
    "tempo_minutos"
]

media = tempos_falha.mean()

mediana = tempos_falha.median()

modas = tempos_falha.mode()

moda = modas.iloc[0]


# ============================================================
# 13. DISTRIBUIÇÃO SUAVIZADA — KDE
# ============================================================

dados_kde = tempos_falha.to_numpy()

kde = gaussian_kde(
    dados_kde
)

x_kde = np.linspace(
    0,
    tempo_total_minutos,
    300
)

y_kde = kde(
    x_kde
)


# ============================================================
# 14. AJUSTAR ALTURA DA KDE
# ============================================================

altura_maxima = (
    df_distribuicao[
        "quantidade_falhas"
    ].max()
)

y_kde = (
    y_kde /
    y_kde.max()
) * altura_maxima


# ============================================================
# 15. GRÁFICO
# ============================================================

fig = go.Figure()


# ============================================================
# 16. BARRAS
# ============================================================

fig.add_trace(
    go.Bar(

        x=centros_bins,

        y=df_distribuicao[
            "quantidade_falhas"
        ],

        width=2.0,

        name="Falhas",

        text=df_distribuicao[
            "quantidade_falhas"
        ],

        textposition="outside",

        customdata=df_distribuicao[
            ["faixa"]
        ],

        hovertemplate=(
            "Classe: %{customdata[0]}"
            "<br>"
            "Falhas: %{y}"
            "<extra></extra>"
        )
    )
)


# ============================================================
# 17. CURVA KDE
# ============================================================

fig.add_trace(
    go.Scatter(

        x=x_kde,

        y=y_kde,

        mode="lines",

        name="Distribuição suavizada (KDE)",

        line=dict(
            width=4,
            shape="spline"
        ),

        hovertemplate=(
            "Tempo: %{x:.2f} min"
            "<br>"
            "Distribuição: %{y:.2f}"
            "<extra></extra>"
        )
    )
)


# ============================================================
# 18. MÉDIA
# ============================================================

fig.add_vline(

    x=media,

    line_dash="dash",

    line_width=3,

    line_color="red",

    annotation_text=(
        f"Média: {media:.2f} min"
    ),

    annotation_position="top"
)


# ============================================================
# 19. MEDIANA
# ============================================================

fig.add_vline(

    x=mediana,

    line_dash="dot",

    line_width=3,

    line_color="green",

    annotation_text=(
        f"Mediana: {mediana:.2f} min"
    ),

    annotation_position="top"
)


# ============================================================
# 20. MODA
# ============================================================

fig.add_vline(

    x=moda,

    line_dash="dashdot",

    line_width=3,

    line_color="blue",

    annotation_text=(
        f"Moda: {moda:.2f} min"
    ),

    annotation_position="bottom"
)


# ============================================================
# 21. LAYOUT
# ============================================================

fig.update_layout(

    title=(
        "Distribuição de falhas "
        "por tempo de trabalho"
    ),

    xaxis=dict(

        title="Tempo de trabalho (minutos)",

        tickmode="array",

        tickvals=centros_bins,

        ticktext=nomes_bins,

        range=[
            0,
            tempo_total_minutos
        ]
    ),

    yaxis=dict(
        title="Quantidade de falhas"
    ),

    template="plotly_white",

    bargap=0.05,

    hovermode="x unified",

    legend=dict(

        orientation="h",

        yanchor="bottom",

        y=1.02,

        xanchor="center",

        x=0.5
    )
)


# ============================================================
# 22. EXIBIR GRÁFICO
# ============================================================

fig.show()


# ============================================================
# 23. EXIBIR CLASSES
# ============================================================

print("\nClasses:")

for i, faixa in enumerate(
    nomes_bins
):

    print(
        f"Bin {i}: {faixa}"
    )


# ============================================================
# 24. EXIBIR CLASSIFICAÇÃO
# ============================================================

print(
    "\nClassificação dos dados:"
)

print(
    df[
        [
            "tempo_minutos",
            "falha",
            "bin",
            "faixa"
        ]
    ]
)


# ============================================================
# 25. ESTATÍSTICAS
# ============================================================

print("\nEstatísticas:")

print(
    f"Média: {media:.2f} minutos"
)

print(
    f"Mediana: {mediana:.2f} minutos"
)

print(
    f"Moda: {moda:.2f} minutos"
)

print(
    f"Total de falhas: "
    f"{len(tempos_falha)}"
)


Classes:
Bin 0: 0-2
Bin 1: 3-5
Bin 2: 6-8
Bin 3: 9-11
Bin 4: 12-14
Bin 5: 15-17
Bin 6: 18-20
Bin 7: 21-20
Bin 8: 21-20
Bin 9: 21-20

Classificação dos dados:
    tempo_minutos  falha  bin  faixa
0             0.0      0    0    0-2
1             1.0      0    0    0-2
2             2.0      1    0    0-2
3             3.0      0    1    3-5
4             4.0      1    1    3-5
5             5.0      0    2    6-8
6             6.0      0    2    6-8
7             7.0      0    3   9-11
8             8.0      1    3   9-11
9             9.0      0    4  12-14
10           10.0      1    4  12-14
11           11.0      1    5  15-17
12           12.0      1    5  15-17
13           13.0      0    6  18-20
14           14.0      1    6  18-20
15           15.0      0    7  21-20
16           16.0      1    7  21-20
17           17.0      1    8  21-20
18           18.0      0    8  21-20
19           19.0      1    9  21-20

Estatísticas:
Média: 11.30 minutos
Mediana: 11.50 minutos
Moda: